# 03. 제로샷 데모 (한 지수) + ARIMA 비교
모델 실행 함수(`run_forecasts`)를 불러 돌리고, 결과는 변수 `fc` 에만 둡니다(파일 저장 없음). 그림은 셀 아래에 바로 뜨니 우클릭으로 복사하세요.

Timestral 가중치가 없으면 `python run.py download-models --sizes 5M 209M` 먼저 (HF 자동 다운로드도 가능).

In [ ]:
import sys, pathlib, warnings
cwd = pathlib.Path.cwd()          # 커널 cwd 가 notebooks/ 든 프로젝트 루트든 상관없이 루트를 찾는다
ROOT = next(p for p in [cwd, *cwd.parents] if (p / "configs").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 100

In [ ]:
TICKER = "KOSPI"
MODELS = ["timestral-209M", "arima", "climatology"]   # timestral-5M/28M/123M/209M, arima, climatology
CTXS = [1024]
EXPERIMENTS = [("m20", 20), ("m5", 5), ("yz20", 20)]   # (시계열, horizon)
ORIGIN_STEP = 20                                         # 데모는 20일 간격 (전체 백테스트는 5)

In [ ]:
from src.eval.backtest import run_forecasts, forecast_latest
from src.eval.summarize import summarize
from src.eval.bands import classify_mdd
from src.viz.plots import select, fan_chart, calibration_plot

fc = run_forecasts(tickers=[TICKER], models=MODELS, experiments=EXPERIMENTS, ctxs=CTXS, origin_step=ORIGIN_STEP)
print(fc.shape); fc.head(3)

In [ ]:
# Fan chart: 분위수 밴드 vs 실현값 (빨간 점 = q05 아래로 이탈)
for series, h in EXPERIMENTS:
    for m in MODELS:
        d = select(fc, model=m, series=series, horizon=h)
        fig = fan_chart(d, f"{m} | {TICKER} {series} h{h}")
        plt.show()

In [ ]:
groups, tau = summarize(fc, origin_step=ORIGIN_STEP)
groups.query("regime == 'all'").sort_values(["series", "horizon", "avg_pinball"])[
    ["model", "series", "horizon", "ctx", "n", "n_eff", "mase_median", "avg_pinball", "coverage80", "ms_per_forecast"]]

In [ ]:
# 꼬리 캘리브레이션: 위반율은 tau 에 가까울수록 좋다 (표본이 작아 p-value 는 참고용)
tau.query("regime == 'all' and tau in [0.05, 0.1]")[
    ["model", "series", "horizon", "tau", "hits", "violation_rate", "pinball", "kupiec_p", "christoffersen_p"]].sort_values(["series", "tau", "pinball"])

In [ ]:
for series, h in EXPERIMENTS:
    fig = calibration_plot(tau, TICKER, series, h)
    plt.show()

## 오늘 기준 밴드 (실전 예측)
가장 최근 날짜를 origin 으로, 정답 없이 앞으로 h일의 분위수만 계산합니다. `m5`/`m20` 은 MDD(음수)라 q05 가 가장 깊은 낙폭입니다.

In [ ]:
LATEST_MODEL = MODELS[0]
latest = forecast_latest(TICKER, LATEST_MODEL, ctx=CTXS[0], experiments=EXPERIMENTS)
latest[["series", "horizon", "origin_date", "y_origin", "q05", "q10", "q50", "q90", "q95"]].round(4)